In [ ]:
!pip install river

In [ ]:
!pip install evidently

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 68.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.7/584.7 kB 23.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 77.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [evidently]


In [ ]:
import river
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from river import datasets
dataset = datasets.Elec2().take(6000)
rows = []
for x,y in dataset:
  row = dict(x)
  row['target'] = y
  row['target'] = row['target']
  rows.append(row)
df = pd.DataFrame(rows)
df['target'] = df['target'].astype('Int64')
df

Uncompressing into /root/river_data/Elec2


,date,day,period,nswprice,nswdemand,vicprice,vicdemand,transfer,target
0,0.000000,2,0.000000,0.056443,0.439155,0.003467,0.422915,0.414912,1
1,0.000000,2,0.021277,0.051699,0.415055,0.003467,0.422915,0.414912,1
2,0.000000,2,0.042553,0.051489,0.385004,0.003467,0.422915,0.414912,1
3,0.000000,2,0.063830,0.045485,0.314639,0.003467,0.422915,0.414912,1
4,0.000000,2,0.085106,0.042482,0.251116,0.003467,0.422915,0.414912,0
...,...,...,...,...,...,...,...,...,...
5995,0.017743,7,0.914894,0.066591,0.326688,0.003467,0.422915,0.414912,1
5996,0.017743,7,0.936170,0.047346,0.348259,0.003467,0.422915,0.414912,0
5997,0.017743,7,0.957447,0.047346,0.357483,0.003467,0.422915,0.414912,0
5998,0.017743,7,0.978723,0.046385,0.370425,0.003467,0.422915,0.414912,0


In [ ]:
df.describe()

,date,day,period,nswprice,nswdemand,vicprice,vicdemand,transfer,target
count,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000,6.000000e+03,6.000000e+03,6.000000e+03,6000.0
mean,0.008068,4.024000,0.500000,0.071439,0.467039,3.467000e-03,4.229150e-01,4.149120e-01,0.394333
std,0.005291,1.989996,0.294778,0.032781,0.158851,2.914578e-16,5.751435e-14,8.271851e-15,0.488748
min,0.000000,1.000000,0.000000,0.040591,0.052812,3.467000e-03,4.229150e-01,4.149120e-01,0.0
25%,0.004425,2.000000,0.250000,0.043983,0.357483,3.467000e-03,4.229150e-01,4.149120e-01,0.0
50%,0.008893,4.000000,0.500000,0.063648,0.493677,3.467000e-03,4.229150e-01,4.149120e-01,0.0
75%,0.013318,6.000000,0.750000,0.087667,0.573825,3.467000e-03,4.229150e-01,4.149120e-01,1.0
max,0.017743,7.000000,1.000000,0.299327,0.935733,3.467000e-03,4.229150e-01,4.149120e-01,1.0


In [ ]:
class SPRTDriftDetector:
    def __init__(self, p0_error_rate=0.12, p1_error_rate=0.25, threshold=4.0):
        self.p0 = p0_error_rate
        self.p1 = p1_error_rate
        self.threshold = threshold


        self.s_error = np.log(p1_error_rate / p0_error_rate)
        self.s_correct = np.log((1 - p1_error_rate) / (1 - p0_error_rate))

        self.T = 0.0
        self.alarms = []
        self.T_history = []

    def update(self, is_error):
        if is_error:
            self.T += self.s_error
        else:
            self.T += self.s_correct

        self.T_history.append(self.T)

        if self.T > self.threshold:
            self.alarms.append(len(self.T_history) - 1)
            self.T = 0.0

        return self.T > self.threshold

In [ ]:
from river import datasets, tree, metrics, stream

dataset = datasets.Elec2().take(6000)


model = tree.HoeffdingTreeClassifier()
metric = metrics.Accuracy()
sprt = SPRTDriftDetector(p0_error_rate=0.12, p1_error_rate=0.18, threshold=4.0)

errors = []
accuracy_history = []
timestamps = []

for i, (x, y) in enumerate(dataset):

    y_pred = model.predict_one(x)


    is_error = int(y_pred != y)
    errors.append(is_error)

    sprt.update(is_error)

    model.learn_one(x, y)

    metric.update(y, y_pred)
    accuracy_history.append(metric.get())
    timestamps.append(i)
print(f"Финальная точность: {metric.get():.3f}")
print(f"Моменты срабатывания SPRT (индексы): {sprt.alarms}")




for i, alarm_idx in enumerate(sprt.alarms):
    start = max(0, alarm_idx - 200)
    end = min(len(errors), alarm_idx + 200)

    error_rate_before = np.mean(errors[max(0, alarm_idx-200):alarm_idx])
    error_rate_after = np.mean(errors[alarm_idx:min(len(errors), alarm_idx+200)])
    acc_drop = accuracy_history[max(0, alarm_idx-50)] - accuracy_history[alarm_idx]

    print(f"Тревога #{i+1} на шаге {alarm_idx}:")
    print(f"Ошибка ДО тревоги (200 шагов): {error_rate_before:.1%}")
    print(f"Ошибка ПОСЛЕ тревоги (200 шагов): {error_rate_after:.1%}")
    print(f"Падение точности в момент тревоги: {acc_drop:.1%}")

Финальная точность: 0.837
Моменты срабатывания SPRT (индексы): [251, 2213, 2444, 2499, 5359, 5374, 5739, 5786, 5801, 5872]
Тревога #1 на шаге 251:
Ошибка ДО тревоги (200 шагов): 19.0%
Ошибка ПОСЛЕ тревоги (200 шагов): 11.5%
Падение точности в момент тревоги: 2.9%
Тревога #2 на шаге 2213:
Ошибка ДО тревоги (200 шагов): 28.0%
Ошибка ПОСЛЕ тревоги (200 шагов): 12.5%
Падение точности в момент тревоги: 0.3%
Тревога #3 на шаге 2444:
Ошибка ДО тревоги (200 шагов): 15.5%
Ошибка ПОСЛЕ тревоги (200 шагов): 15.0%
Падение точности в момент тревоги: 0.5%
Тревога #4 на шаге 2499:
Ошибка ДО тревоги (200 шагов): 24.0%
Ошибка ПОСЛЕ тревоги (200 шагов): 8.0%
Падение точности в момент тревоги: 0.2%
Тревога #5 на шаге 5359:
Ошибка ДО тревоги (200 шагов): 26.5%
Ошибка ПОСЛЕ тревоги (200 шагов): 15.0%
Падение точности в момент тревоги: 0.4%
Тревога #6 на шаге 5374:
Ошибка ДО тревоги (200 шагов): 32.0%
Ошибка ПОСЛЕ тревоги (200 шагов): 9.5%
Падение точности в момент тревоги: 0.4%
Тревога #7 на шаге 5739:
Оши

In [ ]:
from river import drift, datasets, tree, metrics


detector = drift.PageHinkley(min_instances=30, delta=0.005, threshold=50)

model = tree.HoeffdingTreeClassifier()
stream = datasets.Elec2().take(3000)
metric = metrics.Accuracy()

for i, (x, y) in enumerate(stream):
    y_pred = model.predict_one(x)
    error = 1 if y_pred != y else 0
    metric.update(y_pred=y_pred,y_true = y)

    detector.update(error)

    if detector.drift_detected:
        print(f"ДРЕЙФ (PageHinkley) на шаге {i}!")
        detector = drift.PageHinkley(min_instances=30, delta=0.005, threshold=50)

    model.learn_one(x, y)

ДРЕЙФ (PageHinkley) на шаге 1576!
ДРЕЙФ (PageHinkley) на шаге 1856!


In [ ]:
class DDM:

    def __init__(self, min_num_instances=30, warning_level=2.0, drift_level=3.0):
        """
        Args:
            min_num_instances: минимальное количество наблюдений до начала детекции
            warning_level: порог предупреждения (в стандартных отклонениях)
            drift_level: порог дрейфа (в стандартных отклонениях)
        """
        self.min_num_instances = min_num_instances
        self.warning_level = warning_level
        self.drift_level = drift_level


        self.n = 0              # количество наблюдений
        self.n_errors = 0       # количество ошибок
        self.p = 0.0            # текущая частота ошибок
        self.s = 0.0            # текущее стандартное отклонение


        self.p_min = float('inf')
        self.s_min = float('inf')


        self.drift_detected = False
        self.warning_detected = False


        self.history_p = []
        self.history_s = []
        self.history_p_plus_s = []
        self.history_threshold = []

    def add_element(self, error: int):

        self.drift_detected = False
        self.warning_detected = False

        self.n += 1
        self.n_errors += error

        # Текущая частота ошибок и её стандартное отклонение
        self.p = self.n_errors / self.n
        self.s = np.sqrt(self.p * (1 - self.p) / self.n) if self.n > 0 else 0.0

        # Обновляем минимумы
        if self.p + self.s < self.p_min + self.s_min and self.n > self.min_num_instances:
            self.p_min = self.p
            self.s_min = self.s


        self.history_p.append(self.p)
        self.history_s.append(self.s)
        self.history_p_plus_s.append(self.p + self.s)
        self.history_threshold.append(self.p_min + self.drift_level * self.s_min)


        if self.n > self.min_num_instances:
            if self.p + self.s >= self.p_min + self.drift_level * self.s_min:
                self.drift_detected = True
            elif self.p + self.s >= self.p_min + self.warning_level * self.s_min:
                self.warning_detected = True

    def reset(self):
        self.n = 0
        self.n_errors = 0
        self.p = 0.0
        self.s = 0.0
        self.p_min = float('inf')
        self.s_min = float('inf')

In [ ]:
from river import drift, datasets, tree, metrics


detector = DDM()
drifts = []
warnings = []
model = tree.HoeffdingTreeClassifier()
stream = datasets.Elec2().take(6000)
metric = metrics.Accuracy()

for i, (x, y) in enumerate(stream):
    y_pred = model.predict_one(x)
    error = 1 if y_pred != y else 0

    detector.add_element(error)
    metric.update(y_pred,y)

    if detector.warning_detected:
            warnings.append(i)


    if detector.drift_detected:
      drifts.append(i)
      print(f"DRIFT {i}")
      detector.reset()
      model = tree.HoeffdingTreeClassifier()



    model.learn_one(x, y)
print(metric.get())

DRIFT 185
DRIFT 224
DRIFT 337
DRIFT 383
DRIFT 423
DRIFT 602
DRIFT 1777
DRIFT 1860
DRIFT 1993
DRIFT 2060
DRIFT 2118
DRIFT 2486
DRIFT 3119
DRIFT 3150
DRIFT 3262
DRIFT 4396
DRIFT 4555
DRIFT 5108
DRIFT 5183
DRIFT 5228
DRIFT 5328
DRIFT 5388
DRIFT 5613
DRIFT 5658
DRIFT 5821
DRIFT 5995
0.873


In [ ]:
from river import drift, datasets, tree, metrics

ddm = drift.binary.DDM()
model = tree.HoeffdingTreeClassifier()
metric = metrics.Accuracy()

stream = datasets.Elec2().take(6000)

print_warning = True
for i, (x, y) in enumerate(stream):
    y_pred = model.predict_one(x)

    error = int(y_pred != y) if y_pred is not None else 1
    ddm.update(error)


    if ddm.drift_detected:
        print(f"Drift {i}")
        print_warning = True
        ddm = drift.binary.DDM()
        model = tree.HoeffdingTreeClassifier()


    metric.update(y_pred,y)

    # 3. Learn
    model.learn_one(x, y)
print(metric.get())

Drift 185
Drift 224
Drift 337
Drift 383
Drift 423
Drift 602
Drift 1777
Drift 1860
Drift 1993
Drift 2060
Drift 2118
Drift 2486
Drift 3119
Drift 3152
Drift 3229
Drift 3265
Drift 3372
Drift 3406
Drift 4344
Drift 5081
Drift 5183
Drift 5228
Drift 5328
Drift 5388
Drift 5613
Drift 5658
Drift 5821
Drift 5995
0.8738333333333334


In [ ]:
import pandas as pd
from river import drift, datasets, tree, metrics
from evidently import Report
from evidently.presets import DataDriftPreset

model = drift.DriftRetrainingClassifier(
    model=tree.HoeffdingTreeClassifier(),
    drift_detector=drift.binary.DDM(),
    train_in_background=False
)

metric = metrics.Accuracy()
stream = datasets.Elec2().take(6000)


reference_data = []
current_data = []
buffer_size = 1000

for i, (x, y) in enumerate(stream):
    y_pred = model.predict_one(x)


    if y_pred is not None:
        metric.update(y, y_pred)


    row = x.copy()
    row['target'] = y
    if i < buffer_size:
        reference_data.append(row)
    else:
        current_data.append(row)


    if model.drift_detector.drift_detected:
        print(f"Drift detected at index {i}")

    model.learn_one(x, y)

print(f"Final Accuracy: {metric.get()}")

if reference_data and current_data:
    ref_df = pd.DataFrame(reference_data)
    curr_df = pd.DataFrame(current_data)


report = Report(metrics=[DataDriftPreset()])


my_eval = report.run(reference_data=ref_df, current_data=curr_df)


my_eval.save_html("drift_report.html")
print("Evidently report saved as drift_report.html")

Drift detected at index 185
Drift detected at index 225
Drift detected at index 338
Drift detected at index 384
Drift detected at index 424
Drift detected at index 603
Drift detected at index 1778
Drift detected at index 1861
Drift detected at index 1994
Drift detected at index 2061
Drift detected at index 2119
Drift detected at index 2487
Drift detected at index 3120
Drift detected at index 3153
Drift detected at index 3230
Drift detected at index 3266
Drift detected at index 3373
Drift detected at index 3407
Drift detected at index 4345
Drift detected at index 5082
Drift detected at index 5184
Drift detected at index 5229
Drift detected at index 5329
Drift detected at index 5389
Drift detected at index 5614
Drift detected at index 5659
Drift detected at index 5822
Drift detected at index 5996
Final Accuracy: 0.8741456909484914


/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning:

invalid value encountered in divide

/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:3038: RuntimeWarning:

invalid value encountered in divide

/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning:

invalid value encountered in divide

/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:3038: RuntimeWarning:

invalid value encountered in divide



Evidently report saved as drift_report.html
